# MyDigitalTwin — Instagram
**Notebook 03 — Ingestion, exploration, nettoyage → Delta Lake**

Sources :
- `your_instagram_activity/saved/saved_posts.json` → posts sauvegardés
- `your_instagram_activity/comments/post_comments_*.json` → commentaires publics
- `your_instagram_activity/likes/liked_posts.json` → posts likés
- `your_instagram_activity/messages/inbox/*/message_*.json` → métadonnées messages

Outputs :
- `warehouse/instagram_comments`
- `warehouse/instagram_likes`
- `warehouse/instagram_saved`
- `warehouse/instagram_messages_meta`
- `warehouse/instagram_posts_viewed`
- `warehouse/instagram_videos_watched`
- `warehouse/instagram_story_likes`

## Objectifs
- **Clone NLP** : corpus de tes commentaires pour TF-IDF / N-grams
- **K-Means** : activité temporelle (heure, jour)

## 0. Initialisation Spark

In [ ]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import build_spark_session, PROCESSED_DATA, WAREHOUSE

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os
import json
import glob

spark = build_spark_session("MyDigitalTwin - Instagram")
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

# Chemins
IG_ROOT     = os.path.join(PROCESSED_DATA, "INSTAGRAM", "your_instagram_activity")
PARQUET_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "data", "parquet")

---
## PARTIE 1 — Commentaires
### 1.1 Ingestion

In [2]:
# Collecter tous les fichiers post_comments_*.json
comments_files = glob.glob(f"{IG_ROOT}/comments/post_comments_*.json")
print(f"Fichiers commentaires trouvés : {len(comments_files)}")

# Lire et aplatir en Python d'abord (structure trop imbriquée pour spark.read.json)
def parse_comments(files):
    rows = []
    for path in files:
        with open(path, encoding="utf-8", errors="replace") as f:
            data = json.load(f)
        for item in data:
            smd = item.get("string_map_data", {})
            comment = smd.get("Comment", {}).get("value", "")
            owner   = smd.get("Media Owner", {}).get("value", "")
            ts      = smd.get("Time", {}).get("timestamp", 0)
            if comment:
                rows.append({
                    "text":       comment,
                    "media_owner": owner,
                    "timestamp":  ts,
                })
    return rows

comments_rows = parse_comments(comments_files)
print(f"Commentaires extraits : {len(comments_rows):,}")
if comments_rows:
    print("Exemple :", comments_rows[0])

Fichiers commentaires trouvés : 1
Commentaires extraits : 81
Exemple : {'text': 'ð\x9f\x94¥ð\x9f\x94¥', 'media_owner': 'madaclub.be', 'timestamp': 1777100965}


In [3]:
# Créer le DataFrame PySpark
schema_comments = StructType([
    StructField("text",        StringType(), True),
    StructField("media_owner", StringType(), True),
    StructField("timestamp",   LongType(),   True),
])

df_comments = spark.createDataFrame(comments_rows, schema=schema_comments)

# Champs temporels
df_comments = df_comments \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("char_count",    F.length("text")) \
    .withColumn("word_count",    F.size(F.split(F.trim("text"), r"\s+"))) \
    .withColumn("emoji_count",   F.size(F.array_remove(
        F.split(F.regexp_replace("text", r"[\w\s.,!?;:'\"\-()]", " "), " "),
        ""
    ))) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("content_type",  F.lit("comment"))

print(f"Lignes : {df_comments.count():,}")
df_comments.printSchema()
df_comments.show(5, truncate=60)

Lignes : 81
root
 |-- text: string (nullable = true)
 |-- media_owner: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: string (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_weekday: integer (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- word_count: integer (nullable = false)
 |-- emoji_count: integer (nullable = false)
 |-- platform: string (nullable = false)
 |-- content_type: string (nullable = false)

+----------------------------------------+---------------------+----------+-------------------+----------+-----------+----------+-------------+----------+----------+-----------+---------+------------+
|                                    text|          media_owner| timestamp|         event_date|event_year|event_month|event_hour|event_weekday|char_count|word_count|emoji_count| platform|content_type|
+-----------------------

### 1.2 Exploration

In [4]:
print("=== Statistiques texte ===")
df_comments.agg(
    F.avg("char_count").alias("avg_chars"),
    F.avg("word_count").alias("avg_words"),
    F.max("char_count").alias("max_chars"),
).show()

print("\n=== Commentaires par année ===")
df_comments.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Comptes les plus commentés ===")
df_comments.groupBy("media_owner") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

print("\n=== Activité par heure ===")
df_comments.groupBy("event_hour").count().orderBy("event_hour").show()

=== Statistiques texte ===
+-----------------+-----------------+---------+
|        avg_chars|        avg_words|max_chars|
+-----------------+-----------------+---------+
|29.48148148148148|4.777777777777778|      153|
+-----------------+-----------------+---------+


=== Commentaires par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2020|    4|
|      2021|   28|
|      2022|   14|
|      2025|   18|
|      2026|   17|
+----------+-----+


=== Comptes les plus commentés ===
+--------------------+-----+
|         media_owner|count|
+--------------------+-----+
|         madaclub.be|   20|
|        kulturlesite|    7|
|              shades|    5|
|           skusku_fr|    4|
|           xsqueezie|    2|
|            booska_p|    2|
|             arnvudl|    2|
|artstrategyfounda...|    2|
|        brutofficiel|    2|
|  cerveau.artificiel|    2|
+--------------------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|    1|
|         1|    1|
|         3|    1|
|         7|    1|
|         8|    1|
|        10|    2|
|        11|    3|
|        12|    1|
|        14|    7|
|        15|    8|
|        16|    8|
|        17|    6|
|        18|   13|
|        19|    8|
|        20|    4|
|        21|   

In [5]:
print("=== Top 20 mots les plus utilisés ===")
df_comments \
    .withColumn("word", F.explode(F.split(F.lower(F.col("text")), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").rlike(r"[^a-záàâäéèêëîïôùûüç']")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

=== Top 20 mots les plus utilisés ===
+---------+-----+
|     word|count|
+---------+-----+
|      pas|    8|
|      les|    6|
|      mon|    5|
|    c'est|    5|
|     pour|    4|
|      sur|    4|
|      que|    4|
|      est|    3|
|      ton|    3|
|     trop|    3|
|    relou|    3|
|      son|    3|
|     fait|    3|
|  charlie|    3|
|  daronne|    2|
|      des|    2|
| vraiment|    2|
|     sont|    2|
|tellement|    2|
|     reuf|    2|
+---------+-----+



### 1.3 Écriture Parquet

In [ ]:
_path_comments = os.path.join(WAREHOUSE, "instagram_comments")
if not DeltaTable.isDeltaTable(spark, _path_comments):
    df_comments.write.format("delta").save(_path_comments)
else:
    DeltaTable.forPath(spark, _path_comments).alias("t") \
        .merge(df_comments.alias("s"),
               "t.text = s.text AND t.timestamp = s.timestamp") \
        .whenNotMatchedInsertAll() \
        .execute()
print(f"instagram_comments -- {df_comments.count():,} lignes")

---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [7]:
likes_path = f"{IG_ROOT}/likes/liked_posts.json"

def parse_likes(path):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    for item in data:
        ts  = item.get("timestamp", 0)
        url = ""
        # Extraire l'URL du post liké
        for lv in item.get("label_values", []):
            if lv.get("label") == "URL":
                url = lv.get("value", lv.get("href", ""))
                break
        rows.append({"timestamp": ts, "post_url": url})
    return rows

likes_rows = parse_likes(likes_path)
print(f"Likes extraits : {len(likes_rows):,}")
print("Exemple :", likes_rows[0])

Likes extraits : 26,723
Exemple : {'timestamp': 1691597712, 'post_url': 'https://www.instagram.com/reel/CrtP3dUtCxY/'}


In [8]:
schema_likes = StructType([
    StructField("timestamp", LongType(),  True),
    StructField("post_url",  StringType(), True),
])

df_likes = spark.createDataFrame(likes_rows, schema=schema_likes)

df_likes = df_likes \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("action_type",   F.lit("like"))

print(f"Lignes : {df_likes.count():,}")
df_likes.show(5, truncate=60)

Lignes : 26,723
+----------+-------------------------------------------+-------------------+----------+-----------+----------+-------------+---------+-----------+
| timestamp|                                   post_url|         event_date|event_year|event_month|event_hour|event_weekday| platform|action_type|
+----------+-------------------------------------------+-------------------+----------+-----------+----------+-------------+---------+-----------+
|1691597712|https://www.instagram.com/reel/CrtP3dUtCxY/|2023-08-09 16:15:12|      2023|    2023-08|        16|            4|instagram|       like|
|1691597709|   https://www.instagram.com/p/Cuzn198KgCJ/|2023-08-09 16:15:09|      2023|    2023-08|        16|            4|instagram|       like|
|1691597662|   https://www.instagram.com/p/Cq1Dz5ChbL6/|2023-08-09 16:14:22|      2023|    2023-08|        16|            4|instagram|       like|
|1691597639|   https://www.instagram.com/p/Crq9LqvBd5b/|2023-08-09 16:13:59|      2023|    2023-08|   

### 2.2 Exploration

In [9]:
print("=== Likes par année ===")
df_likes.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Likes par heure ===")
df_likes.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Likes par jour de la semaine ===")
df_likes.groupBy("event_weekday").count().orderBy("event_weekday").show()

print("\n=== Mois les plus actifs ===")
df_likes.groupBy("event_month") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Likes par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2020|  382|
|      2021| 3070|
|      2022| 4533|
|      2023| 3358|
|      2024| 4779|
|      2025| 8035|
|      2026| 2566|
+----------+-----+


=== Likes par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  754|
|         1|  437|
|         2|  235|
|         3|  139|
|         4|  210|
|         5|  399|
|         6|  553|
|         7|  646|
|         8|  838|
|         9|  985|
|        10| 1198|
|        11| 1341|
|        12| 1135|
|        13| 1098|
|        14| 1017|
|        15| 1239|
|        16| 1722|
|        17| 1670|
|        18| 1888|
|        19| 1858|
+----------+-----+
only showing top 20 rows


=== Likes par jour de la semaine ===
+-------------+-----+
|event_weekday|count|
+-------------+-----+
|            1| 3855|
|            2| 3830|
|            3| 3762|
|            4| 3830|
|            5| 3923|
|            6| 3821|
|            7| 3702|
+

### 2.3 Écriture Parquet

In [ ]:
_path_likes = os.path.join(WAREHOUSE, "instagram_likes")
if not DeltaTable.isDeltaTable(spark, _path_likes):
    df_likes.write.format("delta").save(_path_likes)
else:
    DeltaTable.forPath(spark, _path_likes).alias("t") \
        .merge(df_likes.alias("s"),
               "t.post_url = s.post_url AND t.timestamp = s.timestamp") \
        .whenNotMatchedInsertAll() \
        .execute()
print(f"instagram_likes -- {df_likes.count():,} lignes")

---
## PARTIE 3 — Messages (métadonnées uniquement)

> **Confidentialité** : on ne stocke PAS le contenu des messages.  
> On extrait uniquement : timestamp, longueur, type de média, conversation anonymisée.

### 3.1 Ingestion

In [11]:
import hashlib

def anonymize(val, salt="mydigitaltwin"):
    return hashlib.sha256(f"{salt}{val}".encode()).hexdigest()[:10]

inbox_root = f"{IG_ROOT}/messages/inbox"
msg_files  = glob.glob(f"{inbox_root}/*/message_*.json")
print(f"Fichiers messages trouvés : {len(msg_files):,}")

def parse_messages(files, my_name="arnvudl"):
    rows_meta = []
    rows_text = []

    for path in files:
        conv_folder = os.path.basename(os.path.dirname(path))
        conv_id     = anonymize(conv_folder)

        with open(path, encoding="utf-8", errors="replace") as f:
            data = json.load(f)

        is_group     = len(data.get("participants", [])) > 2
        participants = len(data.get("participants", []))

        for msg in data.get("messages", []):
            sender_name = msg.get("sender_name", "")
            ts          = msg.get("timestamp_ms", 0)
            content     = msg.get("content", "")
            is_unsent   = msg.get("is_unsent", False)

            # Type de message
            if is_unsent:
                msg_type = "unsent"
            elif msg.get("photos"):
                msg_type = "photo"
            elif msg.get("videos"):
                msg_type = "video"
            elif msg.get("audio_files"):
                msg_type = "audio"
            elif msg.get("share"):
                msg_type = "share"
            elif content:
                msg_type = "text"
            else:
                msg_type = "other"

            # Métadonnées (tout le monde)
            rows_meta.append({
                "conv_id":      conv_id,
                "is_group":     is_group,
                "participants": participants,
                "sender_anon":  anonymize(sender_name),
                "timestamp_ms": ts,
                "msg_type":     msg_type,
                "char_count":   len(content) if content else 0,
            })

            # Texte — uniquement tes messages à toi
            if content and not is_unsent and sender_name == my_name:
                rows_text.append({
                    "text":       content,
                    "timestamp":  ts,
                    "is_group":   is_group,
                    "platform":   "instagram",
                    "content_type": "dm",
                })

    return rows_meta, rows_text


print("Parsing en cours...")
msg_rows, text_rows = parse_messages(msg_files, my_name="arnvudl")
print(f"Messages (métadonnées) : {len(msg_rows):,}")
print(f"Tes messages (texte)   : {len(text_rows):,}")

Fichiers messages trouvés : 459
Parsing en cours...
Messages (métadonnées) : 419,173
Tes messages (texte)   : 0


In [12]:
schema_msgs = StructType([
    StructField("conv_id",      StringType(),  True),
    StructField("is_group",     BooleanType(), True),
    StructField("participants", IntegerType(), True),
    StructField("sender_anon",  StringType(),  True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("msg_type",     StringType(),  True),
    StructField("char_count",   IntegerType(), True),
])

df_msgs = spark.createDataFrame(msg_rows, schema=schema_msgs)

# Timestamp en ms → secondes pour PySpark
df_msgs = df_msgs \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram"))

print(f"Lignes : {df_msgs.count():,}")
df_msgs.printSchema()
df_msgs.show(5)

Lignes : 419,173
root
 |-- conv_id: string (nullable = true)
 |-- is_group: boolean (nullable = true)
 |-- participants: integer (nullable = true)
 |-- sender_anon: string (nullable = true)
 |-- timestamp_ms: long (nullable = true)
 |-- msg_type: string (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: string (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_weekday: integer (nullable = true)
 |-- platform: string (nullable = false)

+----------+--------+------------+-----------+-------------+--------+----------+--------------------+----------+-----------+----------+-------------+---------+
|   conv_id|is_group|participants|sender_anon| timestamp_ms|msg_type|char_count|          event_date|event_year|event_month|event_hour|event_weekday| platform|
+----------+--------+------------+-----------+-------------+--------+----------+--------------------+--

### 3.2 Exploration

In [13]:
print("=== Répartition par type de message ===")
df_msgs.groupBy("msg_type").count().orderBy(F.desc("count")).show()

print("\n=== Groupes vs conversations privées ===")
df_msgs.groupBy("is_group").count().show()

print("\n=== Messages par année ===")
df_msgs.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_msgs.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 10 conversations les plus actives ===")
df_msgs.groupBy("conv_id", "is_group", "participants") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

print("\n=== Longueur moyenne des messages texte ===")
df_msgs.filter(F.col("msg_type") == "text") \
    .agg(
        F.avg("char_count").alias("avg_chars"),
        F.max("char_count").alias("max_chars"),
    ).show()

=== Répartition par type de message ===
+--------+------+
|msg_type| count|
+--------+------+
|    text|387157|
|   other|  8341|
|   audio|  7444|
|   share|  7395|
|   photo|  7152|
|   video|  1677|
|  unsent|     7|
+--------+------+


=== Groupes vs conversations privées ===
+--------+------+
|is_group| count|
+--------+------+
|    true|111115|
|   false|308058|
+--------+------+


=== Messages par année ===


+----------+------+
|event_year| count|
+----------+------+
|      2020|  4440|
|      2021| 13064|
|      2022| 25760|
|      2023| 69094|
|      2024|125088|
|      2025|166469|
|      2026| 15258|
+----------+------+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|10949|
|         1| 4300|
|         2| 1283|
|         3|  455|
|         4|  650|
|         5| 1869|
|         6| 3199|
|         7| 6090|
|         8| 9127|
|         9|13379|
|        10|15763|
|        11|20577|
|        12|20591|
|        13|20811|
|        14|20599|
|        15|23965|
|        16|28401|
|        17|29684|
|        18|33276|
|        19|38354|
+----------+-----+
only showing top 20 rows


=== Top 10 conversations les plus actives ===
+----------+--------+------------+-----+
|   conv_id|is_group|participants|count|
+----------+--------+------------+-----+
|4380c698af|   false|           2|67244|
|43feea1d85|    true|          14|44954|
|f3e010c9df|   fa

### 3.3 Écriture Parquet

In [ ]:
_path_msgs = os.path.join(WAREHOUSE, "instagram_messages_meta")
if not DeltaTable.isDeltaTable(spark, _path_msgs):
    df_msgs.write.format("delta").save(_path_msgs)
else:
    DeltaTable.forPath(spark, _path_msgs).alias("t") \
        .merge(df_msgs.alias("s"),
               "t.conv_id = s.conv_id AND t.sender_anon = s.sender_anon AND t.timestamp_ms = s.timestamp_ms AND t.msg_type = s.msg_type") \
        .whenNotMatchedInsertAll() \
        .execute()
print(f"instagram_messages_meta -- {df_msgs.count():,} lignes")

---
## PARTIE 4 — Saved Posts

Posts sauvegardés = signal d'intérêt fort.  
On conserve le `href` pour pouvoir retrouver le post original.

### 4.1 Ingestion

In [15]:
saved_path = f"{IG_ROOT}/saved/saved_posts.json"

def parse_saved(path):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    for item in data.get("saved_saved_media", []):
        title = item.get("title", "")
        saved_on = item.get("string_map_data", {}).get("Saved on", {})
        href  = saved_on.get("href", "")
        ts    = saved_on.get("timestamp", 0)
        rows.append({
            "account":   title,
            "post_href": href,
            "timestamp": ts,
        })
    return rows

saved_rows = parse_saved(saved_path)
print(f"Posts sauvegardés : {len(saved_rows):,}")
print("Exemple :", saved_rows[0])

Posts sauvegardés : 17
Exemple : {'account': 'usthemob', 'post_href': 'https://www.instagram.com/p/DXZqLgGDC1E/', 'timestamp': 1776792515}


In [16]:
schema_saved = StructType([
    StructField("account",   StringType(), True),
    StructField("post_href", StringType(), True),
    StructField("timestamp", LongType(),   True),
])

df_saved = spark.createDataFrame(saved_rows, schema=schema_saved)

df_saved = df_saved \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("action_type",   F.lit("saved"))

print(f"Lignes : {df_saved.count():,}")
df_saved.show(5, truncate=70)

Lignes : 17
+----------------+-------------------------------------------+----------+-------------------+----------+-----------+----------+-------------+---------+-----------+
|         account|                                  post_href| timestamp|         event_date|event_year|event_month|event_hour|event_weekday| platform|action_type|
+----------------+-------------------------------------------+----------+-------------------+----------+-----------+----------+-------------+---------+-----------+
|        usthemob|   https://www.instagram.com/p/DXZqLgGDC1E/|1776792515|2026-04-21 17:28:35|      2026|    2026-04|        17|            3|instagram|      saved|
|   flo_climatrek|   https://www.instagram.com/p/DXJYKHCDCtB/|1776782813|2026-04-21 14:46:53|      2026|    2026-04|        14|            3|instagram|      saved|
|     eklavya_frr|https://www.instagram.com/reel/DT8RInhkUMy/|1776770588|2026-04-21 11:23:08|      2026|    2026-04|        11|            3|instagram|      saved|
|sam

### 4.2 Exploration

In [17]:
print("=== Posts sauvegardés par année ===")
df_saved.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 15 comptes dont tu sauvegardes le plus ===")
df_saved.groupBy("account") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show(truncate=40)

print("\n=== Activité par heure ===")
df_saved.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Mois les plus actifs ===")
df_saved.groupBy("event_month") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Posts sauvegardés par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2024|    2|
|      2025|    2|
|      2026|   13|
+----------+-----+


=== Top 15 comptes dont tu sauvegardes le plus ===
+--------------------+-----+
|             account|count|
+--------------------+-----+
|            usthemob|    1|
|       flo_climatrek|    1|
|         eklavya_frr|    1|
|    sametgorgozfilms|    1|
|          edelynemia|    1|
|  ibizastardustradio|    1|
|thehybriddesigner.np|    1|
|      brillyondabeat|    1|
|    kellybadakdesign|    1|
|         djmc_gaz974|    1|
|          alyxxcould|    1|
|   livingthedream.wa|    1|
|           fitwcurly|    1|
|       madameb0nplan|    1|
|         viewsfrance|    1|
+--------------------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         7|    1|
|        11|    1|
|        12|    1|
|        14|    3|
|        15|    1|
|        16|    5|
|        17|    2|
|       

### 4.3 Écriture Parquet

In [ ]:
_path_saved = os.path.join(WAREHOUSE, "instagram_saved")
if not DeltaTable.isDeltaTable(spark, _path_saved):
    df_saved.write.format("delta").save(_path_saved)
else:
    DeltaTable.forPath(spark, _path_saved).alias("t") \
        .merge(df_saved.alias("s"),
               "t.post_href = s.post_href AND t.timestamp = s.timestamp") \
        .whenNotMatchedInsertAll() \
        .execute()
print(f"instagram_saved -- {df_saved.count():,} lignes")

In [ ]:
# ── PARTIE 5 — Posts vus (algo Meta) ──────────────────────────────────────────
import glob as _glob

def _find_ig_file(patterns):
    """Trouve un fichier Instagram par liste de patterns glob."""
    for p in patterns:
        matches = _glob.glob(p)
        if matches:
            return matches[0]
    return None

posts_viewed_path = _find_ig_file([
    f"{IG_ROOT}/ads_and_topics/posts_viewed.json",
    f"{IG_ROOT}/impressions/posts_viewed.json",
    f"{IG_ROOT}/your_topics/posts_viewed.json",
    os.path.join(os.path.dirname(IG_ROOT), "ads_information", "ads_and_topics", "posts_viewed.json"),
])

def parse_posts_viewed(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("impressions_history_posts_seen") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

posts_rows = parse_posts_viewed(posts_viewed_path)
print(f"Posts vus : {len(posts_rows):,}")

schema_pv = StructType([
    StructField("author",    StringType(), True),
    StructField("timestamp", LongType(),   True),
])

if posts_rows:
    df_posts_viewed = spark.createDataFrame(posts_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("post_viewed"))

    _path_pv = os.path.join(WAREHOUSE, "instagram_posts_viewed")
    if not DeltaTable.isDeltaTable(spark, _path_pv):
        df_posts_viewed.write.format("delta").save(_path_pv)
    else:
        DeltaTable.forPath(spark, _path_pv).alias("t") \
            .merge(df_posts_viewed.alias("s"),
                   "t.timestamp = s.timestamp") \
            .whenNotMatchedInsertAll() \
            .execute()
    print(f"instagram_posts_viewed -- {df_posts_viewed.count():,} lignes")
else:
    print("⚠ posts_viewed.json introuvable ou vide — fichier skippé")

# ── PARTIE 6 — Vidéos regardées ───────────────────────────────────────────────
videos_watched_path = _find_ig_file([
    f"{IG_ROOT}/ads_and_topics/videos_watched.json",
    f"{IG_ROOT}/impressions/videos_watched.json",
    f"{IG_ROOT}/your_topics/videos_watched.json",
    os.path.join(os.path.dirname(IG_ROOT), "ads_information", "ads_and_topics", "videos_watched.json"),
])

def parse_videos_watched(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("impressions_history_videos_watched") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

videos_rows = parse_videos_watched(videos_watched_path)
print(f"Vidéos regardées : {len(videos_rows):,}")

if videos_rows:
    df_videos_watched = spark.createDataFrame(videos_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("video_watched"))

    _path_vw = os.path.join(WAREHOUSE, "instagram_videos_watched")
    if not DeltaTable.isDeltaTable(spark, _path_vw):
        df_videos_watched.write.format("delta").save(_path_vw)
    else:
        DeltaTable.forPath(spark, _path_vw).alias("t") \
            .merge(df_videos_watched.alias("s"),
                   "t.timestamp = s.timestamp") \
            .whenNotMatchedInsertAll() \
            .execute()
    print(f"instagram_videos_watched -- {df_videos_watched.count():,} lignes")
else:
    print("⚠ videos_watched.json introuvable ou vide — fichier skippé")

# ── PARTIE 7 — Story Likes ────────────────────────────────────────────────────
story_likes_path = _find_ig_file([
    f"{IG_ROOT}/story_activities/story_likes.json",
    f"{IG_ROOT}/story_interactions/story_likes.json",
    f"{IG_ROOT}/likes/story_likes.json",
    f"{IG_ROOT}/story_likes.json",
])

def parse_story_likes(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("story_activities_story_likes") or data.get("story_likes") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

story_rows = parse_story_likes(story_likes_path)
print(f"Story likes : {len(story_rows):,}")

if story_rows:
    df_story_likes = spark.createDataFrame(story_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("story_like"))

    _path_sl = os.path.join(WAREHOUSE, "instagram_story_likes")
    if not DeltaTable.isDeltaTable(spark, _path_sl):
        df_story_likes.write.format("delta").save(_path_sl)
    else:
        DeltaTable.forPath(spark, _path_sl).alias("t") \
            .merge(df_story_likes.alias("s"),
                   "t.timestamp = s.timestamp") \
            .whenNotMatchedInsertAll() \
            .execute()
    print(f"instagram_story_likes -- {df_story_likes.count():,} lignes")
else:
    print("⚠ story_likes.json introuvable ou vide — fichier skippé")

# ── PARTIE 8 — Recherches Instagram ──────────────────────────────────────────
searches_path = _find_ig_file([
    f"{IG_ROOT}/searches/word_or_phrase_searches.json",
    f"{IG_ROOT}/word_or_phrase_searches.json",
    os.path.join(os.path.dirname(IG_ROOT), "logged_information", "recent_searches", "word_or_phrase_searches.json"),
    os.path.join(os.path.dirname(IG_ROOT), "recent_searches", "word_or_phrase_searches.json"),
])

def parse_ig_searches(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data.get("searches_keyword") or data.get("keyword_searches") or (data if isinstance(data, list) else [])
    for item in items:
        smd   = item.get("string_map_data", {})
        query = (smd.get("Recherche", {}) or smd.get("Search", {})).get("value", "") or item.get("title", "")
        ts    = (smd.get("Heure", {}) or smd.get("Time", {})).get("timestamp", 0)
        if ts and query:
            rows.append({"query": query, "timestamp": int(ts)})
    return rows

search_rows = parse_ig_searches(searches_path)
print(f"Recherches Instagram : {len(search_rows):,}")

schema_search = StructType([
    StructField("query",     StringType(), True),
    StructField("timestamp", LongType(),   True),
])

if search_rows:
    df_ig_searches = spark.createDataFrame(search_rows, schema=schema_search) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("search")) \
        .withColumn("char_count",    F.length("query")) \
        .withColumn("word_count",    F.size(F.split(F.trim("query"), r"\s+")))

    _path_igs = os.path.join(WAREHOUSE, "instagram_searches")
    if not DeltaTable.isDeltaTable(spark, _path_igs):
        df_ig_searches.write.format("delta").save(_path_igs)
    else:
        DeltaTable.forPath(spark, _path_igs).alias("t") \
            .merge(df_ig_searches.alias("s"),
                   "t.query = s.query AND t.timestamp = s.timestamp") \
            .whenNotMatchedInsertAll() \
            .execute()
    print(f"instagram_searches -- {df_ig_searches.count():,} lignes")
else:
    print("⚠ word_or_phrase_searches.json introuvable ou vide — fichier skippé")

spark.stop()
print("\n✓ Notebook Instagram terminé.")

In [20]:
spark.stop()